# Microsoft Agent Framework — Azure OpenAI (Responses API)

Trong ví dụ mã này, bạn sẽ sử dụng **Microsoft Agent Framework (MAF)** để tạo một tác nhân đơn giản hỗ trợ bởi **Azure OpenAI** sử dụng **Responses API**.

> **Ghi chú di chuyển:** Ví dụ này trước đây sử dụng Semantic Kernel với GitHub Models. Nó đã được chuyển sang Microsoft Agent Framework, và GitHub Models (đã ngừng phát triển, sẽ nghỉ hưu vào tháng 7 năm 2026) đã được thay thế bằng Azure OpenAI, hỗ trợ Responses API. `OpenAIChatClient` trong MAF hướng tới endpoint ổn định `/openai/v1/` của Azure OpenAI và sử dụng Responses API theo mặc định.

Mục đích của ví dụ này là trình bày các bước sẽ được áp dụng sau này trong các ví dụ mã bổ sung khi triển khai các mẫu tác nhân khác nhau.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Nhập Các Gói Python Cần Thiết


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Định nghĩa một Công cụ

Trong Microsoft Agent Framework, một **công cụ** là một hàm Python đơn giản được trang trí bằng `@tool` mà tác nhân có thể gọi. Dưới đây chúng ta định nghĩa một công cụ trả về một điểm đến kỳ nghỉ ngẫu nhiên và tránh lặp lại điểm đến trước đó.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Tạo tác nhân

Ở đây, chúng tôi tạo một tác nhân có tên `TravelAgent`.

Trong ví dụ này, chúng tôi sử dụng các hướng dẫn rất cơ bản. Bạn có thể thoải mái chỉnh sửa các hướng dẫn này để quan sát cách hành vi của tác nhân thay đổi.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Chạy Agent

Bây giờ chúng ta có thể chạy agent. Chúng ta tạo một `AgentSession` để agent nhớ cuộc trò chuyện qua các lượt, sau đó gửi hai `user_inputs`. Lần đầu tiên yêu cầu một chuyến đi; lần thứ hai nói rằng người dùng không thích đề xuất và yêu cầu một lựa chọn khác — agent sử dụng lịch sử phiên làm việc cùng với công cụ `get_random_destination` để trả lời.

Bạn có thể chỉnh sửa các tin nhắn này để quan sát cách agent phản ứng khác nhau. Các phản hồi được **truyền trực tiếp** từng token một.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
